# Modelagem Matemática, Cinemática e Controle do Robô Otto

## Objetivos da aula

1. Descrever matematicamente a posição de uma perna articulada a partir dos ângulos das juntas (cinemática direta).
2. Modelar o movimento oscilatório das juntas por funções senoidais.
3. Calcular velocidade e aceleração angular e relacioná-las ao torque dos atuadores.
4. Compreender a estrutura de um sistema de controle em malha fechada com controlador PID.
5. Implementar e simular os modelos em software de computação numérica, tais como no MATLAB (ou Octave).

---

## Motivação

Ao montar o [robô Otto](https://www.ottodiy.com/), você construiu um sistema físico com múltiplos atuadores, segmentos rígidos e restrições geométricas. Quando o Otto dá um passo, três fenômenos ocorrem simultaneamente: a **geometria** da estrutura se altera conforme os servos mudam de ângulo; a **física** impõe restrições de inércia e gravidade; e o **controle** reage continuamente, corrigindo desvios entre posição desejada e real.

Esta aula conecta esses três domínios para permitir compreender e diagnosticar quantitativamente o comportamento do robô.


```{note}
**Relação com problemas de engenharia civil**

Os princípios desta aula têm análogos diretos em estruturas e dinâmica:

| Conceito — robótica | Análogo — engenharia civil |
|---|---|
| Cinemática de perna articulada | Geometria de treliça com barras rotacionadas |
| Oscilação senoidal da junta | Vibração livre sob carga harmônica |
| Centro de massa do bípede | Centróide e condição de tombamento |
| Torque gravitacional sobre segmento | Momento fletor por carga excêntrica |
| Controlador PID em malha fechada | Amortecedor ativo sob ação sísmica |

A matemática subjacente é a mesma; o sistema físico é diferente.
```


---

## 1. Cinemática direta de segmentos articulados

### 1.1 Formulação do problema

**Cinemática** descreve o movimento de corpos em termos de posição, velocidade e aceleração, sem considerar as forças causadoras. A **cinemática direta** resolve um problema específico: dados os ângulos de cada junta, qual é a posição do atuador final?

```{note}
Em robôs bípedes como o Otto, o atuador final é o pé. O problema é análogo a determinar a posição dos nós de uma treliça a partir das dimensões e orientações das barras.
```

### 1.2 Um segmento rotacionando

Considere um segmento rígido de comprimento $L$ fixado na origem. Quando forma um ângulo $\theta$ com o eixo horizontal, as coordenadas de sua extremidade livre são:

$$
x = L\cos(\theta), \qquad y = L\sin(\theta)
$$

O seno e o cosseno emergem da decomposição de um vetor rotacionado sobre dois eixos ortogonais — fundamento de toda a cinemática de sistemas articulados em cadeia aberta.


In [3]:
from IPython.display import HTML, display
import uuid

canvas_id = f"canvas_{uuid.uuid4().hex}"

display(HTML(f"""
<canvas id="{canvas_id}" width="600" height="600"
        style="border:1px solid #ccc;border-radius:10px;background:white;"></canvas>
<script>
(function() {{
    const canvas = document.getElementById("{canvas_id}");
    const ctx = canvas.getContext("2d");
    const L = 180, cx = 300, cy = 300;
    let theta = 0;
    function draw() {{
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.strokeStyle = "#e0e0e0"; ctx.lineWidth = 1;
        for(let i=0;i<=600;i+=50){{
            ctx.beginPath(); ctx.moveTo(i,0); ctx.lineTo(i,600); ctx.stroke();
            ctx.beginPath(); ctx.moveTo(0,i); ctx.lineTo(600,i); ctx.stroke();
        }}
        ctx.strokeStyle="#999"; ctx.lineWidth=2;
        ctx.beginPath(); ctx.moveTo(0,cy); ctx.lineTo(600,cy); ctx.stroke();
        ctx.beginPath(); ctx.moveTo(cx,0); ctx.lineTo(cx,600); ctx.stroke();
        const x = cx + L*Math.cos(theta);
        const y = cy - L*Math.sin(theta);
        ctx.strokeStyle="#1565c0"; ctx.lineWidth=6;
        ctx.beginPath(); ctx.moveTo(cx,cy); ctx.lineTo(x,y); ctx.stroke();
        ctx.fillStyle="#1565c0";
        ctx.beginPath(); ctx.arc(cx,cy,10,0,2*Math.PI); ctx.fill();
        ctx.fillStyle="red";
        ctx.beginPath(); ctx.arc(x,y,12,0,2*Math.PI); ctx.fill();
        ctx.fillStyle="#222"; ctx.font="18px Arial";
        ctx.fillText("θ = "+(theta*180/Math.PI).toFixed(1)+"°", 20, 30);
        theta += 0.005;
        requestAnimationFrame(draw);
    }}
    draw();
}})();
</script>
"""))


#### 1.2.1 Implementacao no MATLAB

O arquivo `Ex1_2_1_CinPerna.m` plota a extremidade de um segmento rotacionando conforme o angulo varia -- verificacao direta de $x = L\cos\theta$, $y = L\sin\theta$.

[Baixar `Ex1_2_1_CinPerna.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex1_2_1_CinPerna.m)


In [4]:
import os
import matlab.engine

eng = matlab.engine.start_matlab()

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
matlab_path = os.path.join(repo_root, "matlab")

eng.eval(f"addpath(genpath('{matlab_path}'));", nargout=0)

In [5]:
eng.Ex1_2_1_CinPerna(nargout=0)


### 1.3 Dois segmentos em série — modelo de quadril e joelho

A perna do Otto é modelada como dois segmentos rígidos em série (comprimentos $L_1$ e $L_2$). O primeiro gira com ângulo $\theta_1$ em relação ao referencial global; o segundo gira com $\theta_2$ em relação ao primeiro.

Posição da junta intermediária:

$$
x_1 = L_1\cos(\theta_1), \qquad y_1 = L_1\sin(\theta_1)
$$

Posição do atuador final (pé), por acumulação de rotações:

$$
\boxed{x_2 = L_1\cos(\theta_1) + L_2\cos(\theta_1 + \theta_2)}
$$

$$
\boxed{y_2 = L_1\sin(\theta_1) + L_2\sin(\theta_1 + \theta_2)}
$$

O ângulo acumulado $(\theta_1 + \theta_2)$ é a orientação absoluta do segundo segmento. Para uma cadeia de $n$ segmentos, o ângulo absoluto da $i$-ésima junta generaliza-se para $\Theta_i = \sum_{k=1}^{i} \theta_k$.


In [6]:
from IPython.display import HTML
HTML("""<div style="border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;">
<p style="font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;">Simulador — Cinemática direta de dois segmentos</p>
<canvas id="legCanvas" width="540" height="300" style="display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;"></canvas>
<div style="max-width:540px;margin:0.75rem auto 0;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:100px;color:#333;">θ₁ (1ª junta)</span>
    <input type="range" id="th1" min="-70" max="70" value="30" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:36px;color:#1a7a50;" id="th1val">30°</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;">
    <span style="font-size:13px;font-family:monospace;min-width:100px;color:#333;">θ₂ (2ª junta)</span>
    <input type="range" id="th2" min="-70" max="70" value="20" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:36px;color:#2563a8;" id="th2val">20°</span>
  </div>
  <p style="font-size:12px;font-family:monospace;color:#666;text-align:center;margin-top:8px;" id="footpos">Calculando...</p>
</div></div>
<script>
(function(){
  var cv=document.getElementById('legCanvas'), ctx=cv.getContext('2d');
  var L=110, cx=270, cy=120;
  function draw(){
    var t1=parseFloat(document.getElementById('th1').value)*Math.PI/180;
    var t2=parseFloat(document.getElementById('th2').value)*Math.PI/180;
    document.getElementById('th1val').textContent=Math.round(t1*180/Math.PI)+'°';
    document.getElementById('th2val').textContent=Math.round(t2*180/Math.PI)+'°';
    var x1=cx+L*Math.cos(t1), y1=cy-L*Math.sin(t1);
    var x2=x1+L*Math.cos(t1+t2), y2=y1-L*Math.sin(t1+t2);
    ctx.clearRect(0,0,540,300);
    ctx.strokeStyle='#e8e8e8'; ctx.lineWidth=0.5;
    for(var i=0;i<=540;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,300);ctx.stroke();}
    for(var j=0;j<=300;j+=40){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(540,j);ctx.stroke();}
    ctx.strokeStyle='#bbb'; ctx.lineWidth=1;
    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(540,cy);ctx.stroke();
    ctx.beginPath();ctx.moveTo(cx,0);ctx.lineTo(cx,300);ctx.stroke();
    ctx.strokeStyle='#1a7a50'; ctx.lineWidth=5; ctx.lineCap='round';
    ctx.beginPath();ctx.moveTo(cx,cy);ctx.lineTo(x1,y1);ctx.stroke();
    ctx.strokeStyle='#2563a8'; ctx.lineWidth=5;
    ctx.beginPath();ctx.moveTo(x1,y1);ctx.lineTo(x2,y2);ctx.stroke();
    [[cx,cy,'#1a7a50',7],[x1,y1,'#888',5],[x2,y2,'#c0392b',7]].forEach(function(p){
      ctx.beginPath();ctx.arc(p[0],p[1],p[3],0,2*Math.PI);ctx.fillStyle=p[2];ctx.fill();
    });
    ctx.font='12px monospace'; ctx.fillStyle='#1a7a50';
    ctx.fillText('θ₁='+Math.round(t1*180/Math.PI)+'°',cx+8,cy-10);
    ctx.fillStyle='#2563a8';
    ctx.fillText('θ₂='+Math.round(t2*180/Math.PI)+'°',x1+8,y1-8);
    ctx.fillStyle='#c0392b'; ctx.font='bold 12px monospace';
    ctx.fillText('pé (x₂, y₂)',x2+8,y2+4);
    var rx=((x2-cx)/L).toFixed(3), ry=((-(y2-cy))/L).toFixed(3);
    document.getElementById('footpos').textContent='x₂ = '+rx+' L    y₂ = '+ry+' L   (normalizado por L)';
  }
  document.getElementById('th1').addEventListener('input',draw);
  document.getElementById('th2').addEventListener('input',draw);
  draw();
})();
</script>""")


#### 1.3.1 Implementacao no MATLAB -- cinematica estatica

O arquivo `Ex1_3_1_DoisSegm.m` calcula e plota a configuracao da perna para angulos fixos. Altere `theta1` e `theta2` e observe o deslocamento do pe.

[Baixar `Ex1_3_1_DoisSegm.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex1_3_1_DoisSegm.m)

> **Aprofundamento na Aula 2:** cinematica 3D com `rigidBodyTree` (Robotics System Toolbox). [Ver Aula 2](#)


In [7]:
eng.Ex1_3_1_DoisSegm(nargout=0)


### 1.4 Modelo cinemático simplificado do Otto bípede

O Otto difere de um manipulador planar clássico: seus movimentos distribuem-se em **dois planos independentes**, controlados por dois pares de servos.

- **Servos do quadril** (superiores): rotação lateral de cada perna em plano aproximadamente paralelo ao piso.
- **Servos dos pés** (inferiores): inclinação da base de apoio, alterando a altura efetiva do corpo e redistribuindo a carga entre as pernas.

#### Movimento lateral do quadril

O deslocamento lateral da perna é:

$$x = L \sin(\theta_h)$$

onde $L$ é o comprimento da perna (quadril ao piso) e $\theta_h$ o ângulo de rotação lateral. Durante a caminhada, o quadril oscila senoidalmente:

$$\theta_h(t) = A \sin(\omega t)$$

Essa variação transfere o peso de uma perna para a outra, criando a condição para o avanço do passo.

#### Movimento de inclinação dos pés

Uma aproximação para a altura vertical do corpo é:

$$y = h + r \sin(\theta_f)$$

onde $h$ é a altura neutra, $r$ um parâmetro geométrico do pé e $\theta_f$ o ângulo do servo. Com os dois pés oscilando em oposição de fase, o robô produz inclinações controladas que complementam o deslocamento lateral.

#### Combinação dos movimentos e visualização 3D

A caminhada resulta da **sincronização entre quatro servos**: os do quadril impõem deslocamento lateral do centro de massa (CoM), enquanto os dos pés controlam inclinação e transferência de carga. O Otto comporta-se como um sistema bípede oscilatório de baixa complexidade mecânica — uma aproximação do **pêndulo invertido**.

Como os gráficos planos (x–t ou y–t isolados) não capturam a combinação simultânea dos movimentos lateral e vertical, o simulador abaixo projeta a trajetória do CoM no espaço tridimensional (x lateral, y vertical, t temporal), tornando a coordenação entre os quatro servos visualmente legível.


In [8]:
from IPython.display import HTML
HTML("""
<div style="border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;">
<p style="font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;">
  Simulador 3D — Trajetória do CoM durante a caminhada do Otto
</p>
<canvas id="otto3d" width="540" height="360"
  style="display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;"></canvas>
<div style="max-width:540px;margin:0.6rem auto 0;display:flex;gap:16px;flex-wrap:wrap;">
  <div style="display:flex;align-items:center;gap:8px;">
    <span style="font-size:12px;font-family:monospace;color:#333;">A quadril (°)</span>
    <input type="range" id="ottA" min="5" max="25" value="15" style="width:90px;">
    <span style="font-size:12px;font-family:monospace;color:#1a7a50;" id="ottAV">15</span>
  </div>
  <div style="display:flex;align-items:center;gap:8px;">
    <span style="font-size:12px;font-family:monospace;color:#333;">r pé</span>
    <input type="range" id="ottR" min="1" max="10" value="4" style="width:90px;">
    <span style="font-size:12px;font-family:monospace;color:#2563a8;" id="ottRV">4</span>
  </div>
  <div style="display:flex;align-items:center;gap:8px;">
    <span style="font-size:12px;font-family:monospace;color:#333;">ω (rad/s)</span>
    <input type="range" id="ottW" min="1" max="5" value="2" step="0.5" style="width:90px;">
    <span style="font-size:12px;font-family:monospace;color:#7a3a8a;" id="ottWV">2</span>
  </div>
</div>
<p style="font-size:11px;font-family:monospace;color:#888;text-align:center;margin-top:6px;">
  Eixos: horizontal = deslocamento lateral (x) &nbsp;|&nbsp; vertical = altura do CoM (y) &nbsp;|&nbsp; profundidade = tempo (t)
</p>
</div>

<script>
(function(){
  var cv = document.getElementById('otto3d');
  var ctx = cv.getContext('2d');
  var W=540, H=360;

  // Projeção isométrica simplificada
  function project(x, y, z){
    // x=lateral, y=altura, z=tempo (profundidade)
    var px = W/2 + x*3.5 - z*1.8;
    var py = H/2 - y*4   - z*0.9;
    return {px:px, py:py};
  }

  var trail = [];
  var T_MAX = 60;  // pontos da trilha

  var t0 = 0;
  var L = 5, h = 0;

  function getParams(){
    var A = parseFloat(document.getElementById('ottA').value)*Math.PI/180;
    var r = parseFloat(document.getElementById('ottR').value);
    var w = parseFloat(document.getElementById('ottW').value);
    document.getElementById('ottAV').textContent = Math.round(A*180/Math.PI);
    document.getElementById('ottRV').textContent = r;
    document.getElementById('ottWV').textContent = w;
    return {A:A, r:r, w:w};
  }

  function draw(){
    var p = getParams();
    t0 += 0.04;

    // Gera ponto atual
    var th_h = p.A * Math.sin(p.w * t0);
    var th_f = p.A * Math.sin(p.w * t0 + Math.PI/2);
    var x = L * Math.sin(th_h);
    var y = h + p.r * Math.sin(th_f);
    var tNorm = (t0 % (2*Math.PI/p.w)) / (2*Math.PI/p.w) * 30 - 15;

    trail.push({x:x, y:y, z:tNorm, t:t0});
    if(trail.length > T_MAX) trail.shift();

    ctx.clearRect(0,0,W,H);

    // Grade de fundo (plano xz)
    ctx.strokeStyle='#eeeeee'; ctx.lineWidth=0.5;
    for(var xi=-20;xi<=20;xi+=5){
      var a=project(xi,h,-15), b=project(xi,h,15);
      ctx.beginPath();ctx.moveTo(a.px,a.py);ctx.lineTo(b.px,b.py);ctx.stroke();
    }
    for(var zi=-15;zi<=15;zi+=5){
      var a=project(-20,h,zi), b=project(20,h,zi);
      ctx.beginPath();ctx.moveTo(a.px,a.py);ctx.lineTo(b.px,b.py);ctx.stroke();
    }

    // Eixos
    ctx.lineWidth=1.2;
    // eixo x (lateral)
    ctx.strokeStyle='#1a7a50';
    var ox=project(0,h,0), ex=project(22,h,0);
    ctx.beginPath();ctx.moveTo(ox.px,ox.py);ctx.lineTo(ex.px,ex.py);ctx.stroke();
    ctx.fillStyle='#1a7a50';ctx.font='bold 11px monospace';
    ctx.fillText('x (lateral)',ex.px+2,ex.py);
    // eixo y (altura)
    ctx.strokeStyle='#2563a8';
    var ey=project(0,h+18,0);
    ctx.beginPath();ctx.moveTo(ox.px,ox.py);ctx.lineTo(ey.px,ey.py);ctx.stroke();
    ctx.fillStyle='#2563a8';
    ctx.fillText('y (altura)',ey.px+2,ey.py-2);
    // eixo z (tempo)
    ctx.strokeStyle='#7a3a8a';
    var ez=project(0,h,-17);
    ctx.beginPath();ctx.moveTo(ox.px,ox.py);ctx.lineTo(ez.px,ez.py);ctx.stroke();
    ctx.fillStyle='#7a3a8a';
    ctx.fillText('t (tempo)',ez.px-30,ez.py+12);

    // Trilha 3D
    if(trail.length > 1){
      for(var i=1;i<trail.length;i++){
        var pp=project(trail[i-1].x,trail[i-1].y,trail[i-1].z);
        var cp=project(trail[i].x,trail[i].y,trail[i].z);
        var alpha = i/trail.length;
        ctx.strokeStyle='rgba(197,56,56,'+alpha.toFixed(2)+')';
        ctx.lineWidth = 1+alpha*1.5;
        ctx.beginPath();ctx.moveTo(pp.px,pp.py);ctx.lineTo(cp.px,cp.py);ctx.stroke();
      }
    }

    // Ponto atual do CoM
    var cp = project(x,y,tNorm);
    ctx.beginPath();ctx.arc(cp.px,cp.py,6,0,2*Math.PI);
    ctx.fillStyle='#c0392b';ctx.fill();

    // Projeção no plano do piso (sombra)
    var sp = project(x,h,tNorm);
    ctx.beginPath();ctx.arc(sp.px,sp.py,3,0,2*Math.PI);
    ctx.fillStyle='rgba(160,160,160,0.5)';ctx.fill();
    ctx.strokeStyle='rgba(160,160,160,0.4)';ctx.lineWidth=0.8;
    ctx.beginPath();ctx.moveTo(cp.px,cp.py);ctx.lineTo(sp.px,sp.py);ctx.stroke();

    ctx.fillStyle='#444';ctx.font='11px monospace';
    ctx.fillText('x='+x.toFixed(2)+' cm   y='+y.toFixed(2)+' cm', 8, H-8);

    requestAnimationFrame(draw);
  }

  draw();
})();
</script>
""")


```{note}
**Interpretação do gráfico 3D:** a curva vermelha traça a trajetória do CoM no espaço (x, y, t). A projeção cinza no piso mostra o deslocamento lateral puro. Observe que nem o gráfico de x(t) isolado nem o de y(t) isolado revela a elipse tridimensional que o CoM percorre — apenas a combinação dos dois movimentos sincronizados produz o padrão de caminhada.
```

```{note}
**Observação cinemática importante:** o Otto não levanta as pernas como um humanoide. O avanço resulta da inclinação do corpo pelos servos dos pés, da transferência lateral de peso pelo quadril e da alteração da geometria de apoio que permite o deslizamento ou arraste do pé. Os movimentos assemelham-se mais a um **balanço dinâmico controlado** do que a uma marcha antropomórfica clássica.
```


#### 1.4.1 Implementacao no MATLAB -- oscilacao lateral do quadril

O arquivo `Ex1_4_1_OscLat.m` simula $x(t) = L\sin(A\sin(\omega t))$ e plota o deslocamento lateral do centro de massa em dois paineis.

[Baixar `Ex1_4_1_OscLat.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex1_4_1_OscLat.m)


In [9]:
eng.Ex1_4_1_OscLat(nargout=0)


---

## 2. Movimento no tempo — modelagem da caminhada

### 2.1 Funções senoidais como modelos de movimento oscilatório

A caminhada não é uma sequência de posições estáticas. Os ângulos das juntas variam continuamente de forma periódica, modelados por:

$$
\theta(t) = A \sin(\omega t + \varphi)
$$

| Parâmetro | Unidade | Significado físico | Efeito na caminhada |
|---|---|---|---|
| $A$ | graus | Amplitude — deflexão angular máxima | Comprimento do passo |
| $\omega$ | rad/s | Frequência angular | Cadência |
| $\varphi$ | rad | Fase inicial | Sincronismo entre pernas |

O mesmo modelo aparece em dinâmica estrutural (vibrações de pontes), hidráulica (oscilação de marés) e engenharia sísmica (acelerograma simplificado).

### 2.2 O papel do defasamento de fase

As duas pernas oscilam com **mesma amplitude e mesma frequencia**, mas com fases distintas:

$$
\theta_{\text{esq}}(t) = A\sin(\omega t), \qquad \theta_{\text{dir}}(t) = A\sin(\omega t + \varphi)
$$

O parametro $\varphi$ determina a **coordenacao entre as pernas**:

| Defasamento $\varphi$ | Comportamento | Resultado |
|---|---|---|
| $0$ | Pernas em fase | Robo oscila sem avancar |
| $\pi/2$ | Pernas em quadratura -- quando uma esta no maximo, a outra esta na posicao neutra | Padrao de caminhada estavel |
| $\pi$ | Pernas em oposicao de fase | Marcha rigida, sem transferencia suave de carga |

```{note}
**Analogia e contraste com engenharia civil:** em estruturas sob cargas harmonicas, defasamentos geram *batimentos* -- amplificacoes periodicas que podem causar fadiga. O fenomeno fisico e o mesmo; a diferenca e a **intencao**: em estruturas o defasamento e um problema a evitar; no bipede, ele e a solucao para a coordenacao do passo.
```

O simulador abaixo mostra as **duas pernas simultaneamente**. Mova $\varphi$ e observe quando o robo "anda" ($\varphi \approx \pi/2$) e quando "trava" ($\varphi \approx 0$).


In [10]:
from IPython.display import HTML
HTML("""
<div style="border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;">
<p style="font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;">Simulador -- Defasamento entre pernas (fi)</p>
<canvas id="wCanvas2" width="540" height="220" style="display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;"></canvas>
<div style="max-width:540px;margin:0.75rem auto 0;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:120px;color:#333;">A -- amplitude</span>
    <input type="range" id="wAmp2" min="5" max="50" value="20" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:44px;color:#1a7a50;" id="wAmpV2">20 graus</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:120px;color:#333;">omega -- freq. angular</span>
    <input type="range" id="wFreq2" min="1" max="8" value="2" step="0.5" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:44px;color:#2563a8;" id="wFreqV2">2 rad/s</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;">
    <span style="font-size:13px;font-family:monospace;min-width:120px;color:#333;">fi -- defasagem</span>
    <input type="range" id="wPhi2" min="0" max="628" value="157" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:60px;color:#7a3a8a;" id="wPhiV2">0.50pi rad</span>
  </div>
  <p style="font-size:11px;color:#888;margin-top:8px;font-family:monospace;text-align:center;" id="wStatus2">...</p>
</div></div>
<script>
(function(){
  var cv=document.getElementById("wCanvas2"),ctx=cv.getContext("2d");
  var W=540,H=220,cy=H/2,t0=0;
  function draw(){
    var A=parseFloat(document.getElementById("wAmp2").value);
    var w=parseFloat(document.getElementById("wFreq2").value);
    var phi=parseFloat(document.getElementById("wPhi2").value)/100;
    document.getElementById("wAmpV2").textContent=A+"gr";
    document.getElementById("wFreqV2").textContent=w+" rad/s";
    document.getElementById("wPhiV2").textContent=(phi/Math.PI).toFixed(2)+"pi rad";
    var pn=phi/Math.PI,st,sc;
    if(pn<0.1){st="fi~0: pernas em fase -- robo oscila sem avancar";sc="#c0392b";}
    else if(Math.abs(pn-0.5)<0.12){st="fi~pi/2: quadratura -- padrao de caminhada estavel";sc="#1a7a50";}
    else if(Math.abs(pn-1.0)<0.12){st="fi~pi: oposicao de fase -- marcha rigida";sc="#555";}
    else{st="fi = "+pn.toFixed(2)+"pi rad";sc="#555";}
    var el=document.getElementById("wStatus2");
    el.textContent=st;el.style.color=sc;
    ctx.clearRect(0,0,W,H);
    ctx.strokeStyle="#ebebeb";ctx.lineWidth=0.5;
    for(var i=0;i<W;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,H);ctx.stroke();}
    for(var j=0;j<H;j+=30){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(W,j);ctx.stroke();}
    ctx.strokeStyle="#bbb";ctx.lineWidth=1;
    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(W,cy);ctx.stroke();
    var scl=(H/2-14)/55;
    ctx.beginPath();ctx.strokeStyle="#1a7a50";ctx.lineWidth=2.2;
    for(var px=0;px<W;px++){var t=(px/W)*12+t0;var v=A*Math.sin(w*t);var py=cy-v*scl;if(px===0)ctx.moveTo(px,py);else ctx.lineTo(px,py);}
    ctx.stroke();
    ctx.beginPath();ctx.strokeStyle="#c0392b";ctx.lineWidth=2.2;ctx.setLineDash([7,4]);
    for(var px=0;px<W;px++){var t=(px/W)*12+t0;var v=A*Math.sin(w*t+phi);var py=cy-v*scl;if(px===0)ctx.moveTo(px,py);else ctx.lineTo(px,py);}
    ctx.stroke();ctx.setLineDash([]);
    ctx.font="11px monospace";
    ctx.fillStyle="#1a7a50";ctx.fillRect(8,6,18,3);ctx.fillText("perna esquerda  th_esq = A sin(omega t)",30,14);
    ctx.fillStyle="#c0392b";for(var d=8;d<28;d+=10)ctx.fillRect(d,22,6,3);
    ctx.fillText("perna direita    th_dir = A sin(omega t + fi)",30,28);
    t0+=0.008;requestAnimationFrame(draw);
  }
  draw();
})();
</script>
""")

#### 2.1.1 Implementacao no MATLAB -- posicao, velocidade e aceleracao angular

O arquivo `Ex2_3_1_SenCinem.m` plota $\theta(t)$, $\dot{\theta}(t)$ e $\ddot{\theta}(t)$ para $A = 20^\circ$, $\omega = 2$ rad/s e $\varphi = \pi/4$.

[Baixar `Ex2_3_1_SenCinem.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex2_3_1_SenCinem.m)

```{note}
A amplitude da aceleracao angular e $A\omega^2 = 80^\circ/\text{s}^2$, evidenciando o crescimento **quadratico** da demanda de torque com a frequencia de caminhada.
```

> **Aprofundamento na Aula 2:** modelo dinamico completo incluindo atrito -- identificacao parametrica do servo real. [Ver Aula 2](#)


In [11]:
eng.Ex2_3_1_SenCinem(nargout=0)


### 2.4 Dinamica rotacional e torque

O servo precisa produzir torque suficiente para vencer a inercia rotacional do segmento:

$$
\tau = I\,\alpha
$$

Para $\theta(t) = A\sin(\omega t)$, a aceleracao angular e $\alpha(t) = -A\omega^2\sin(\omega t)$, resultando em torque maximo:

$$
\tau_{\max} = I\,A\omega^2
$$

**Por que o torque cresce com $\omega^2$?** Dobrar a frequencia de caminhada exige inverter o sentido do movimento duas vezes mais rapido. Como a forca necessaria para reverter um movimento circular cresce com o quadrado da velocidade angular, o torque exigido ao servo aumenta quatro vezes. Para o SG90 (limite: 0,177 N.m), isso impoe um **teto fisico de cadencia**: acima de certa frequencia o servo nao consegue acompanhar a trajetoria senoidal e o robo comeca a perder passos.

A gravidade impoe ainda um torque adicional:

$$
\tau_g = m\,g\,r\,\cos(\theta)
$$

onde $r$ e a distancia do centro de massa ao eixo de rotacao. **Quando o torque gravitacional e dominante?** Quando o segmento esta proximo da horizontal ($\theta \approx 0$), pois $\cos(0) = 1$ e maximo -- exatamente como o momento fletor e maximo no centro de uma viga horizontal com carga vertical. Para segmentos mais verticais ($\theta \approx 90^\circ$), o torque dinamico domina porque $\cos(90^\circ) \approx 0$.

O simulador abaixo mostra essa competicao em tempo real. Aumente $\omega$ e observe o momento em que o torque total ultrapassa o limite do SG90.


In [12]:
from IPython.display import HTML
HTML("""
<div style="border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;">
<p style="font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;">Simulador -- Torque dinamico vs gravitacional (SG90)</p>
<canvas id="tqCanvas" width="540" height="210" style="display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;"></canvas>
<div style="max-width:540px;margin:0.6rem auto 0;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:130px;">omega (rad/s)</span>
    <input type="range" id="tqW" min="0.5" max="8" value="2" step="0.5" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:44px;color:#1a7a50;" id="tqWV">2.0</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:130px;">A -- amplitude (graus)</span>
    <input type="range" id="tqA" min="5" max="40" value="20" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:44px;color:#2563a8;" id="tqAV">20</span>
  </div>
  <p style="font-size:11px;font-family:monospace;text-align:center;margin-top:6px;" id="tqStatus">--</p>
</div></div>
<script>
(function(){
  var cv=document.getElementById("tqCanvas"),ctx=cv.getContext("2d");
  var W=540,H=210,t0=0;
  var m=0.010,g=9.81,r=0.025,L=0.05,I=m*L*L/3,SG90=0.177;
  function draw(){
    var w=parseFloat(document.getElementById("tqW").value);
    var A=parseFloat(document.getElementById("tqA").value)*Math.PI/180;
    document.getElementById("tqWV").textContent=w.toFixed(1);
    document.getElementById("tqAV").textContent=Math.round(A*180/Math.PI);
    t0+=0.025;
    var theta=A*Math.sin(w*t0);
    var alpha=-A*w*w*Math.sin(w*t0);
    var tDin=Math.abs(I*alpha);
    var tGrav=Math.abs(m*g*r*Math.cos(theta));
    var tTot=tDin+tGrav;
    ctx.clearRect(0,0,W,H);
    ctx.fillStyle="#f9f9f9";ctx.fillRect(0,0,W,H);
    var barH=140,barY=30,barX=60,maxV=SG90*1.5,scl=barH/maxV;
    var limY=barY+barH-SG90*scl;
    ctx.strokeStyle="#e74c3c";ctx.lineWidth=1.5;ctx.setLineDash([6,3]);
    ctx.beginPath();ctx.moveTo(barX,limY);ctx.lineTo(barX+380,limY);ctx.stroke();
    ctx.setLineDash([]);
    ctx.fillStyle="#e74c3c";ctx.font="bold 10px monospace";
    ctx.fillText("Limite SG90: 0.177 N.m",barX+384,limY+4);
    var bW=60,gap=30;
    var bars=[
      {val:tDin,color:"#2563a8",label:"tau din."},
      {val:tGrav,color:"#1a7a50",label:"tau grav."},
      {val:tTot,color:"#7a3a8a",label:"tau total"}
    ];
    var sx=barX+20;
    bars.forEach(function(b,i){
      var bX=sx+i*(bW+gap);
      var bH=Math.min(b.val*scl,barH);
      var col=b.val>SG90?"#e74c3c":b.color;
      ctx.fillStyle=col;
      ctx.fillRect(bX,barY+barH-bH,bW,bH);
      ctx.fillStyle="#444";ctx.font="11px monospace";
      ctx.fillText(b.label,bX,barY+barH+16);
      ctx.fillStyle=col;ctx.font="bold 10px monospace";
      ctx.fillText((b.val*1000).toFixed(1)+" mN.m",bX,barY+barH-bH-5);
    });
    var ok=tTot<=SG90;
    var el=document.getElementById("tqStatus");
    el.textContent=ok?"OK: dentro da capacidade do SG90":"ATENCAO: torque excede o limite do SG90!";
    el.style.color=ok?"#1a7a50":"#e74c3c";
    requestAnimationFrame(draw);
  }
  draw();
  ["tqW","tqA"].forEach(function(id){
    document.getElementById(id).addEventListener("input",draw);
  });
})();
</script>
""")

#### 2.4.1 Implementacao no MATLAB -- torque dinamico e gravitacional

O arquivo `Ex2_4_1_Torque.m` calcula e plota $\tau_{\text{din}}$, $\tau_{\text{grav}}$ e $\tau_{\text{total}}$ ao longo do tempo, comparando o pico com a capacidade nominal do servo SG90 (0,177 N.m). Compativel com Octave (nao usa toolboxes).

[Baixar `Ex2_4_1_Torque.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex2_4_1_Torque.m)

> **Aprofundamento na Aula 2:** modelo com atrito -- identificacao experimental do momento de inercia real do servo SG90. [Ver Aula 2](#)


In [13]:
eng.Ex2_4_1_Torque(nargout=0)


---

## 3. Controle em malha fechada

### 3.1 A necessidade do controle

Um servo real nao responde como um dispositivo ideal: apresenta **atraso de resposta**, **sobressinal** (*overshoot*) e **erro em regime estacionario** causado por atrito ou nao linearidades. Sem correcao, esses desvios se acumulam ao longo da caminhada e o robo perde equilibrio.

O controlador em malha fechada soluciona esse problema comparando continuamente a posicao real com a referencia e calculando a acao corretiva necessaria.

```{note}
**Glossario de termos de controle** -- referencia rapida para engenheiros civis:

| Termo | Significado |
|---|---|
| *Setpoint* | Valor de referencia desejado -- o angulo-alvo do servo |
| *Overshoot* | Ultrapassagem: a saida excede momentaneamente o setpoint antes de estabilizar |
| Regime estacionario | Comportamento apos os transientes se extinguirem |
| Funcao de transferencia | Razao saida/entrada no dominio da frequencia ($s$) -- analogia: funcao de rigidez dinamica |
| Malha fechada / aberta | Com / sem realimentacao da saida para correcao da entrada |
| Constante de tempo $\tau$ | Tempo para o sistema atingir 63 pct do valor final |
```

### 3.2 Estrutura da malha de controle

```
r(t) -->[Sum]--> C(s) --> G(s) --> y(t)
          |-                        |
          +-------- H(s) <----------+
```

- $r(t)$: referencia -- angulo desejado;
- $y(t)$: saida -- angulo real medido pelo sensor;
- $e(t) = r(t) - y(t)$: sinal de erro;
- $C(s)$: controlador PID;
- $G(s)$: planta (modelo do servo);
- $H(s) = 1$: sensor unitario.

#### 3.2.1 O controlador PID -- $C(s)$

A lei de controle do PID e:

$$
u(t) = K_p\,e(t) + K_i\int_0^t e(\tau)\,d\tau + K_d\,\frac{de(t)}{dt}
$$

**Termo proporcional ($K_p$):** acao corretiva proporcional ao erro atual. Ganho elevado reduz o erro rapidamente, mas pode induzir oscilacoes.

**Termo integral ($K_i$):** acumula o erro e elimina o erro em regime estacionario -- analogo a um ajuste progressivo que cresce ate o desvio ser nulo.

**Termo derivativo ($K_d$):** reage a taxa de variacao do erro, reduzindo a acao corretiva quando o erro diminui rapidamente para evitar o *overshoot* -- funcionalmente equivalente a um amortecedor viscoso em sistemas estruturais.

```{note}
**Para esta aula, o objetivo e apenas observar o comportamento qualitativo do PID por meio do simulador.** A teoria formal de projeto de controladores -- funcoes de transferencia, lugar das raizes, margens de fase -- sera tratada em aula dedicada.
```

#### 3.2.2 Modelo do servo -- $G(s)$

O comportamento dinamico do servo e aproximado por um sistema de primeira ordem:

$$
G(s) = \frac{1}{\tau s + 1}
$$

Para $\tau = 0{,}1$ s, o servo leva aproximadamente $5\tau = 0{,}5$ s para atingir 99 pct do valor final em resposta a um degrau. A funcao de transferencia em malha fechada com PID e:

$$
T(s) = \frac{C(s)\,G(s)}{1 + C(s)\,G(s)}
$$

### 3.3 Simulador interativo -- resposta do servo com PID

Ajuste os ganhos e observe: $K_p$ alto gera resposta rapida mas oscilatoria; $K_i$ elimina o erro residual; $K_d$ reduz o overshoot.


In [14]:
from IPython.display import HTML
HTML("""<div style="border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;">
<p style="font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;">
  Simulador — Resposta do servo com controlador PID
</p>
<p style="font-size:12px;color:#555;margin:0 0 0.75rem;">
  O sinal <strong style="color:#1a7a50;">verde</strong> é o <em>setpoint</em> (referência desejada r(t) = 20°·sin(2t)).
  O sinal <strong style="color:#2563a8;">azul</strong> é a resposta real do servo y(t).
  O sinal <strong style="color:#c0392b;">vermelho</strong> é o erro e(t) = r(t) − y(t).
  Ajuste os ganhos e observe: Kp alto → resposta rápida mas oscilatória;
  Ki elimina erro residual; Kd reduz o overshoot.
</p>
<canvas id="pidCanvas" width="540" height="220"
  style="display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;"></canvas>
<div style="max-width:540px;margin:0.75rem auto 0;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:60px;color:#333;">Kp</span>
    <input type="range" id="pKp" min="0.1" max="5" value="1.2" step="0.1" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:36px;color:#1a7a50;" id="pKpV">1.2</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
    <span style="font-size:13px;font-family:monospace;min-width:60px;color:#333;">Ki</span>
    <input type="range" id="pKi" min="0" max="2" value="0.3" step="0.05" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:36px;color:#2563a8;" id="pKiV">0.30</span>
  </div>
  <div style="display:flex;align-items:center;gap:10px;">
    <span style="font-size:13px;font-family:monospace;min-width:60px;color:#333;">Kd</span>
    <input type="range" id="pKd" min="0" max="1" value="0.15" step="0.05" style="flex:1;">
    <span style="font-size:13px;font-family:monospace;min-width:36px;color:#7a3a8a;" id="pKdV">0.15</span>
  </div>
  <p style="font-size:11px;color:#888;margin-top:8px;font-family:monospace;text-align:center;">
    Verde = setpoint r(t) &nbsp;|&nbsp; Azul = saída y(t) &nbsp;|&nbsp; Vermelho = erro e(t)
  </p>
</div>
</div>
<script>
(function(){
  var cv=document.getElementById('pidCanvas'), ctx=cv.getContext('2d');
  var W=540, H=220, cy=H/2;
  var t0=0;
  function simulate(Kp,Ki,Kd){
    var dt=0.02, N=Math.floor(16/dt);  // janela temporal mais larga
    var ref=[],out=[],err=[];
    var y=0,yi=0,ep=0;
    for(var i=0;i<N;i++){
      var t=i*dt, r=20*Math.sin(1.5*t), e=r-y;  // ω=1.5 para curva mais lenta
      yi+=e*dt;
      var de=(e-ep)/dt, u=Kp*e+Ki*yi+Kd*de;
      y+=u*dt*3; y*=0.92;
      ref.push(r); out.push(y); err.push(e); ep=e;
    }
    return {ref:ref, out:out, err:err, N:N};
  }
  function drawCurve(ctx, arr, d, color, shift){
    ctx.beginPath(); ctx.strokeStyle=color; ctx.lineWidth=1.8;
    for(var p=0;p<W;p++){
      var idx=(Math.floor(p/W*d.N)+shift)%d.N;
      var py=cy-arr[idx]*(H/2-10)/28;
      if(p===0)ctx.moveTo(p,py); else ctx.lineTo(p,py);
    }
    ctx.stroke();
  }
  function draw(){
    var Kp=parseFloat(document.getElementById('pKp').value);
    var Ki=parseFloat(document.getElementById('pKi').value);
    var Kd=parseFloat(document.getElementById('pKd').value);
    document.getElementById('pKpV').textContent=Kp.toFixed(1);
    document.getElementById('pKiV').textContent=Ki.toFixed(2);
    document.getElementById('pKdV').textContent=Kd.toFixed(2);
    var d=simulate(Kp,Ki,Kd);
    ctx.clearRect(0,0,W,H);
    ctx.strokeStyle='#ebebeb'; ctx.lineWidth=0.5;
    for(var i=0;i<W;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,H);ctx.stroke();}
    for(var j=0;j<H;j+=30){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(W,j);ctx.stroke();}
    ctx.strokeStyle='#bbb'; ctx.lineWidth=1;
    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(W,cy);ctx.stroke();
    // Banda do setpoint (±2°)
    var sc=(H/2-10)/28;
    ctx.fillStyle='rgba(26,122,80,0.07)';
    ctx.fillRect(0, cy-2*sc, W, 4*sc);
    var shift=Math.floor(t0*30)%d.N;
    drawCurve(ctx, d.ref, d, '#1a7a50', shift);
    drawCurve(ctx, d.out, d, '#2563a8', shift);
    drawCurve(ctx, d.err, d, '#c0392b', shift);
    // Rótulos laterais
    ctx.font='10px monospace'; ctx.fillStyle='#1a7a50';
    ctx.fillText('setpoint', 4, cy - 28*sc - 4);
    ctx.fillStyle='#2563a8';
    ctx.fillText('saída', 4, 16);
    t0 += 0.012;  // velocidade reduzida
    requestAnimationFrame(draw);
  }
  draw();
  ['pKp','pKi','pKd'].forEach(function(id){
    document.getElementById(id).addEventListener('input',draw);
  });
})();
</script>""")


---

## 4. Implementacao no MATLAB

### 4.1 Cinematica dinamica -- animacao da perna

O arquivo `Ex4_1_1_CinAnim.m` combina os resultados das Secoes 1 e 2: avalia a cinematica direta quadro a quadro com angulos variando senoidalmente no tempo, gerando uma animacao da perna em movimento.

[Baixar `Ex4_1_1_CinAnim.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex4_1_1_CinAnim.m)

O codigo abaixo exibe a logica principal do arquivo (execute localmente no MATLAB/Octave):

```matlab
# Ex4_1_1_CinAnim.m -- Animacao da perna do Otto
# Compativel com MATLAB e Octave. Nao requer toolboxes.
L1 = 0.05; L2 = 0.05;   %% comprimentos dos segmentos (m)
A  = 20*pi/180;          %% amplitude (rad)
w  = 2;                  %% frequencia angular (rad/s)
t  = linspace(0, 4*pi/w, 200);

figure; axis equal; axis([-0.12 0.12 -0.12 0.12]); grid on; hold on;
for k = 1:length(t)
    th1 = A*sin(w*t(k));
    th2 = A*sin(w*t(k) + pi/2);
    x1 = L1*cos(th1); y1 = L1*sin(th1);
    x2 = x1 + L2*cos(th1+th2); y2 = y1 + L2*sin(th1+th2);
    cla;
    plot([0 x1 x2],[0 y1 y2],'b-o','LineWidth',3,'MarkerSize',8);
    title(sprintf('t = %.2f s', t(k))); drawnow;
end
```

> **Aprofundamento na Aula 2:** ajuste formal de PID via `pidTuner` (Control System Toolbox). [Ver Aula 2](#)

### 4.2 Modelo Simulink -- malha de controle

Monte no Simulink a seguinte cadeia de blocos:

| Bloco | Parametros | Descricao |
|---|---|---|
| **Sine Wave** | Amplitude: 20, Frequencia: 2 rad/s, Fase: 0 | Referencia $r(t)$ |
| **Sum** | Sinais: `+-` | Calcula $e(t) = r(t) - y(t)$ |
| **PID Controller** | Kp = 1.2, Ki = 0.3, Kd = 0.15 | Controlador |
| **Transfer Fcn** | Numerador: `[1]`, Denominador: `[0.1 1]` | Modelo do servo $G(s)$ |
| **Scope** | 2 entradas | Visualizacao de $r(t)$ e $y(t)$ |

O canal de realimentacao conecta a saida da Transfer Fcn a entrada negativa do Sum.

```{note}
Apos 10 s de simulacao, observe no Scope: o **atraso de fase** entre referencia e resposta, o **overshoot** nos picos e o comportamento em regime estacionario.
```

#### 4.2.1 Implementacao MATLAB/Octave -- resposta ao degrau em malha fechada

O arquivo `Ex4_2_1_MalhaFechada.m` simula numericamente (metodo de Euler) o sistema em malha fechada sem exigir o Control System Toolbox, sendo **compativel com Octave**.

[Baixar `Ex4_2_1_MalhaFechada.m` no GitHub](https://github.com/UniRobotica/robotica/blob/main/matlab/Aula1_CinematicaOtto/Ex4_2_1_MalhaFechada.m)

<!-- ### 4.3 Control System Toolbox

A analise formal com `tf`, `pid`, `feedback` e `stepinfo` -- incluindo calculo de tempo de subida, tempo de acomodacao e overshoot -- sera tratada na **Aula 3: Projeto de Controladores**.
-->

## 5. Síntese

Esta aula introduziu três camadas de descrição matemática de um sistema robótico articulado:

1. **Cinemática direta.** A posição do pé resulta de transformações trigonométricas encadeadas: $\mathbf{p} = \sum_{i=1}^{n} L_i [\cos\Theta_i,\, \sin\Theta_i]^\top$, onde $\Theta_i = \sum_{k=1}^{i}\theta_k$.

2. **Dinâmica do movimento.** O ângulo de cada junta segue $\theta(t) = A\sin(\omega t + \varphi)$. O torque dinâmico máximo cresce com $\omega^2$, impondo limites à cadência de caminhada.

3. **Controle em malha fechada.** O controlador PID minimiza continuamente $e(t) = r(t) - y(t)$ por três ações complementares: proporcional (resposta imediata), integral (eliminação do erro residual) e derivativa (amortecimento do *overshoot*).

---
-->

## Exercicios

```{note}
Os exercicios marcados com **(M/O)** podem ser resolvidos em MATLAB **ou Octave**. A sintaxe e identica na maior parte dos casos. Diferencas relevantes sao indicadas abaixo de cada enunciado.
```

**1.** Calcule analiticamente a posicao do pe para $\theta_1 = 45°$, $\theta_2 = -30°$, $L_1 = L_2 = 5$ cm. Verifique com o simulador interativo da Secao 1.3.

**2. (M/O)** No arquivo `Ex2_3_1_SenCinem.m`, altere $\omega$ de 2 para 4 rad/s. Qual a variacao percentual na amplitude da aceleracao angular? Confirme com a expressao $\alpha_{\max} = A\omega^2$.

**3. (M/O)** Configure $K_p = 5$, $K_i = 0$, $K_d = 0$ no simulador PID e registre o comportamento. Em seguida, acrescente $K_d = 0.5$ e compare. Explique fisicamente a reducao do *overshoot*.

> No Octave, o pacote `control` implementa `pid` e `feedback` com a mesma interface do MATLAB. Instale com `pkg install -forge control` e carregue com `pkg load control`.

**4. (M/O)** O torque maximo do micro servo SG90 e 1,8 kgf.cm = 0,177 N.m. Modele um segmento de perna como haste uniforme de 5 cm e 10 g ($I = mL^2/3$). Para $A = 20°$ e $\omega = 4$ rad/s, o torque dinamico maximo excede a capacidade do servo?

**5.** Em um sistema de amortecimento ativo estrutural, identifique os equivalentes de $r(t)$, $y(t)$, $e(t)$ e $u(t)$ do diagrama de malha fechada (Secao 3.2). Discuta o papel de $K_d$ como amortecedor ativo.
